In [1]:
from datetime import datetime

import gymnasium as gym
import numpy as np
import torch
from torch.utils.tensorboard import SummaryWriter

from src.agents.cart_ppo import PPOCartAgent
from src.utils import run_tests

In [2]:
LR = 2.5e-3
num_episodes = 3000
K_EPOCH = 4
REPEAT = 1
num_tests=1000

In [3]:
run_name = f"mountain_car_ppo_lr{LR}_ne{num_episodes}_k{K_EPOCH}_r{REPEAT}_{datetime.now():%Y%m%d_%H%M%S}"
writer = SummaryWriter(f"./logs/{run_name}")

In [4]:
env = gym.make("MountainCar-v0", render_mode=None)
agent = PPOCartAgent(env=env, learning_rate=LR)

In [ ]:
# F(s, s') = gamma * phi(s') - phi(s)

C = .5
def phi (s):
    position, v = s[0], s[1]
    return np.sin(3 * position) + C * v**2

def reward_bonus (state, next_state, gamma):
    return gamma * phi(next_state) - phi(state)

In [ ]:
SOME_CONSTANT = 1.1
for e in range(num_episodes):
    state, _ = env.reset()

    done = False
    states = []


    # FIRST PASS:
    actions = []
    probs = []
    rewards = []
    running_batch = []
    t = 0

    # DEBUG
    bonuses = 0
    vanilla_rewards = 0
    episode_max_position = state[0]
    episode_min_position = state[0]
    while not done:
        bonus = 0
        states.append(state)

        action, prob = agent.get_action(state)

        total_reward = 0

        next_state, reward, terminated, truncated, _ = env.step(action)

        # HEURISTICS
        if next_state[0] > episode_max_position:
            bonus += SOME_CONSTANT
            episode_max_position = next_state[0]

        if next_state[0] < episode_min_position:
            bonus += SOME_CONSTANT
            episode_min_position = next_state[0]

        bonus += reward_bonus(state, next_state, agent.discount_factor)
        
        bonuses += bonus
        vanilla_rewards += reward

        reward += bonus

        t += 1
        
        total_reward += reward

        done = terminated or truncated

        probs.append(prob)
        actions.append(action)
        rewards.append(total_reward)

        if agent.update_critic(running_batch):
            running_batch = []

        running_batch.append((state, action, total_reward, next_state, done))
        state = next_state



    # TRAINING PASS:
    old_log_probs = torch.stack(probs).detach()
    states_t = torch.tensor(np.array(states), dtype=torch.float32) 
    actions_t = torch.tensor(np.array(actions), dtype=torch.int8) 
        
    entropies = []
    for _ in range(K_EPOCH):
        with torch.no_grad():
            critiques = agent.critic(states_t).squeeze(-1)
        new_log_probs, entropy = agent.evaluate_action(states_t, actions_t)
        ratio = torch.exp(new_log_probs - old_log_probs)
        agent.update(rewards, (ratio, entropy), critiques)
        entropies.append(entropy.detach().numpy())

    writer.add_scalar("reward/entropy", np.average(entropies), e+1)
    writer.add_scalar("reward/total", np.average(rewards), e+1)
    writer.add_scalar("reward/bonus", bonuses, e+1)
    writer.add_scalar("reward/clean", vanilla_rewards, e+1)

print('Complete')

Complete


In [7]:
test_env = gym.make("MountainCar-v0", render_mode=None)
test_env = gym.wrappers.RecordEpisodeStatistics(test_env, buffer_length=num_tests)
test_agent = PPOCartAgent(env, 0, 0)
test_agent.policy.load_state_dict(agent.policy.state_dict())
avg = run_tests(test_agent, test_env, writer, num_tests)
print("avg: ", avg)

avg:  -200.0


In [8]:
dummy_input = torch.randn(1, env.observation_space.shape[0])
torch.onnx.export(
    agent.policy,
    dummy_input,
    f"./data/latest.onnx",
    input_names=["obs"],
    output_names=["action_probs"],
    dynamic_axes={"obs": {0: "batch"}, "action_probs": {0: "batch"}},
    external_data=False,
)
print(f"./data/{run_name}.onnx")

/var/folders/n6/4mwsyk8d3sgcs0_g5vsnhzd00000gn/T/ipykernel_91158/2425558547.py:2: UserWarning: Exporting a model while it is in training mode. Please ensure that this is intended, as it may lead to different behavior during inference. Calling model.eval() before export is recommended.
  torch.onnx.export(
/var/folders/n6/4mwsyk8d3sgcs0_g5vsnhzd00000gn/T/ipykernel_91158/2425558547.py:2: UserWarning: # 'dynamic_axes' is not recommended when dynamo=True, and may lead to 'torch._dynamo.exc.UserError: Constraints violated.' Supply the 'dynamic_shapes' argument instead if export is unsuccessful.
  torch.onnx.export(
W0823 22:25:16.654000 91158 torch/onnx/_internal/exporter/_registration.py:107] torchvision is not installed. Skipping torchvision::nms
W0823 22:25:16.655000 91158 torch/onnx/_internal/exporter/_registration.py:107] torchvision is not installed. Skipping torchvision::roi_align
W0823 22:25:16.656000 91158 torch/onnx/_internal/exporter/_registration.py:107] torchvision is not insta

[torch.onnx] Obtain model graph for `MLP([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `MLP([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decompositions...
[torch.onnx] Run decompositions... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
[torch.onnx] Optimize the ONNX graph...
[torch.onnx] Optimize the ONNX graph... ✅
./data/mountain_car_ppo_lr0.0025_ne3000_k4_r1_20260823_222330.onnx


/opt/homebrew/Caskroom/miniconda/base/lib/python3.13/copyreg.py:99: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)
